# Training-data generation

How synthetic `(source, reuse, script)` triples are produced, and what to check
before trusting a corpus. Runs on CPU with no resources: the substitution
source is a mock, which is enough to exercise the pipeline but not to train a
model anyone should believe.


In [1]:
import sys
sys.path.insert(0, ".")

from retexo import Scriba, normalize
from retexo.datasets.generation import (
    GenerationConfig,
    MockSubstitutionSource,
    SyntheticGenerator,
    Difficulty,
)

scriba = Scriba()

## Seeds

Passages the corpus reuses. In a real run these come from training folds only,
so that no held-out pair contributes even indirectly.


In [2]:
SEEDS = [
    "arma uirumque cano Troiae qui primus ab oris",
    "uox faucibus haesit et inter ruborem atque pallorem",
    "ingentes animos angusto in pectore uersant",
    "obstipui steteruntque comae et uox faucibus haesit",
    "at regina graui iamdudum saucia cura uulnus alit uenis",
    "sunt lacrimae rerum et mentem mortalia tangunt",
]
for s in SEEDS:
    print(" ", s)

  arma uirumque cano Troiae qui primus ab oris
  uox faucibus haesit et inter ruborem atque pallorem
  ingentes animos angusto in pectore uersant
  obstipui steteruntque comae et uox faucibus haesit
  at regina graui iamdudum saucia cura uulnus alit uenis
  sunt lacrimae rerum et mentem mortalia tangunt


## Generating

The sampler draws an operation count from the curriculum, then operations by
weight. It emits one operation per changed position and never expresses a
substitution as a deletion plus an insertion, so the script it produces is the
cheapest one relating the pair rather than merely *a* script relating them.


In [3]:
generator = SyntheticGenerator(
    MockSubstitutionSource(),
    GenerationConfig(variants_per_seed=15, seed=42),
)
examples, report = generator.generate(SEEDS)
print(report.summary())

seeds=6 attempted=90 kept=90 (100.0%)  build_fail=0 verify_fail=0 trivial_dropped=0 trivially_recoverable=0 canonicalized=0 disagreed=0


### What came out

In [4]:
for example in examples[:4]:
    print("SRC:", " ".join(example["source_tokens"]))
    print("VAR:", " ".join(example["target_tokens"]))
    print("OPS:", example["script"].op_counts(),
          f"cost={example['script'].cost():.2f}")
    print()

SRC: arma uirumque cano Troiae qui primus ab oris
VAR: arma-infl uirumque-ant cano-spec Troiae qui primus ab oris primus-ins
OPS: {'MORPH': 1, 'ANT': 1, 'HYPO': 1, 'NOP': 5, 'INS': 1} cost=3.00

SRC: arma uirumque cano Troiae qui primus ab oris
VAR: arma uirumque cano Troiae qui primus ab-infl oris
OPS: {'NOP': 7, 'MORPH': 1} cost=0.25

SRC: arma uirumque cano Troiae qui primus ab oris
VAR: arma uirumque cano Troiae-sim qui primus ab oris
OPS: {'NOP': 7, 'SYN-DIST': 1} cost=0.60

SRC: arma uirumque cano Troiae qui primus ab oris
VAR: arma uirumque cano Troiae-ne qui primus ab oris
OPS: {'NOP': 7, 'NE-SUB': 1} cost=0.75



## Check 1 — every example replays and inverts

Scriba shares no code with the builder, so agreement is a real check. A
corpus that fails here is unusable, and it is cheaper to find out now than
after training.


In [5]:
bad_replay = bad_invert = 0
for example in examples:
    script, source, target = example["script"], example["source_tokens"], example["target_tokens"]
    if not scriba.verify(script, source, target):
        bad_replay += 1
    restored = scriba.execute(script.invert(), target)
    if [normalize(t) for t in restored] != [normalize(t) for t in source]:
        bad_invert += 1

print(f"replay failures    : {bad_replay}/{len(examples)}")
print(f"inversion failures : {bad_invert}/{len(examples)}")
assert bad_replay == 0 and bad_invert == 0

replay failures    : 0/90
inversion failures : 0/90


## Check 2 — the operation mix

Operations are drawn by weight rather than uniformly. A uniform draw over the
full inventory would produce variants dominated by rare operations and unlike
any attested reuse. In a real run these weights are anchored to the frequencies
the oracle finds on genuine pairs.


In [6]:
import pandas as pd

mix = pd.Series(report.op_mix).sort_values(ascending=False)
mix_pct = (mix / mix.sum() * 100).round(1)
pd.DataFrame({"count": mix, "share %": mix_pct})

,count,share %
NOP,440,62.2
MORPH,81,11.5
SYN,52,7.4
DEL,37,5.2
INS,32,4.5
SYN-DIST,21,3.0
HYPER,15,2.1
HYPO,14,2.0
ANT,9,1.3
NE-SUB,6,0.8


## Check 3 — difficulty is recorded, not filtered

Every example carries its difficulty signals. Discarding on any one of them
would be irreversible and the thresholds are uncalibrated, so the corpus keeps
everything and leaves the choice to a curriculum, a loss weighting, or a filter
applied later.

`oracle_recoverable` is the signal that corresponds to AlphaGeometry's
auxiliary-construction filter, and it stays `None` until the oracle's resources
are available.


In [7]:
rows = []
for example in examples:
    d = example["difficulty"]
    rows.append({
        "n_ops": d.n_operations,
        "n_subs": d.n_substitutions,
        "overlap": round(d.lexical_overlap, 2),
        "positional_recoverable": d.positional_recoverable,
        "oracle_recoverable": d.oracle_recoverable,
    })
difficulty = pd.DataFrame(rows)
difficulty.describe().loc[["mean", "min", "max"]].round(2)

,n_ops,n_subs,overlap
mean,7.86,2.2,0.65
min,6.00,0.0,0.17
max,10.00,5.0,1.00


In [8]:
share = difficulty.positional_recoverable.mean()
print(f"trivially recoverable: {share:.1%}")
print()
print("AlphaGeometry keeps ~10% of generated theorems, discarding the rest as")
print("solvable without help. Our figure is near zero, but that is an artifact:")
print("a mock replacement is lexically unrelated to the token it replaces, so no")
print("baseline can recover it and everything looks hard. This number carries no")
print("information until the lexical substitution source is in place.")

trivially recoverable: 0.0%

AlphaGeometry keeps ~10% of generated theorems, discarding the rest as
solvable without help. Our figure is near zero, but that is an artifact:
a mock replacement is lexically unrelated to the token it replaces, so no
baseline can recover it and everything looks hard. This number carries no
information until the lexical substitution source is in place.


## Check 4 — the difficulty dial

Operation count is the curriculum. Cost should rise with it; if it does not,
the sampler is not doing what the curriculum assumes.


In [9]:
costs = pd.DataFrame({
    "n_ops": [e["difficulty"].n_operations for e in examples],
    "cost": [e["script"].cost() for e in examples],
    "overlap": [e["difficulty"].lexical_overlap for e in examples],
})
costs.groupby("n_ops").agg(n=("cost", "size"),
                           mean_cost=("cost", "mean"),
                           mean_overlap=("overlap", "mean")).round(2)

,n,mean_cost,mean_overlap
n_ops,,,
6,10,1.44,0.51
7,25,1.49,0.68
8,29,1.77,0.66
9,20,2.28,0.66
10,6,2.41,0.69


## What the model sees

The same script, rendered for each formulation. They are three views of one
supervision, not three datasets.


In [10]:
from retexo.formulations.encoding import ScriptEncoder

example = examples[0]
model_input, target = ScriptEncoder.to_seq2seq(example["script"])
print("seq2seq input :", model_input)
print("seq2seq target:", target)
print()
labels = ScriptEncoder.to_token_labels(example["script"])
print(f"{'reuse token':<18} {'operation':<10} source")
for token, op, source in zip(labels["tokens"], labels["op_labels"], labels["source_indices"]):
    print(f"{token:<18} {op:<10} {source}")

seq2seq input : source: 0:arma 1:uirumque 2:cano 3:Troiae 4:qui 5:primus 6:ab 7:oris reuse: arma-infl uirumque-ant cano-spec Troiae qui primus ab oris primus-ins
seq2seq target: <MORPH> 0>0 arma-infl | <ANT> 1>1 uirumque-ant | <HYPO> 2>2 cano-spec | <NOP> 3>3 Troiae | <NOP> 4>4 qui | <NOP> 5>5 primus | <NOP> 6>6 ab | <NOP> 7>7 oris | <INS> ->8 primus-ins

reuse token        operation  source
arma-infl          MORPH      0
uirumque-ant       ANT        1
cano-spec          HYPO       2
Troiae             NOP        3
qui                NOP        4
primus             NOP        5
ab                 NOP        6
oris               NOP        7
primus-ins         INS        -1


## Known limits of this corpus

1. **Mock substitutions.** Replacements are suffixed forms, not real synonyms.
   Realistic content needs the lexical resources.
2. **The difficulty filter is untestable** for the reason printed above.
3. **A circularity to avoid.** If substitutions are drawn from the same
   resource the oracle detects with, every synthetic example is oracle-
   recoverable by construction and the corpus teaches the model nothing the
   oracle does not already know. Generation must draw on a *wider* source than
   detection, or synthetic pretraining has no purchase.
